In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks
# Obtener la ruta del directorio raíz del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))
# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(PROJECT_ROOT)

INPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'inputs')
OUTPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'outputs')
DOCUMENTS_FOLDER = os.path.join(PROJECT_ROOT, 'documents', 'inputs', 'test_files')

def get_file_from_path(bank:str, file_path: str) -> str:
    return os.path.join(DOCUMENTS_FOLDER, bank, file_path)

file_path = get_file_from_path('Nu', 'nudeb_ago.pdf')

In [2]:
from models import DocumentProcessingFacade
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

doc_processor = DocumentProcessingFacade(file_path)
statement_properties = doc_processor.get_statement_properties() 
bank = statement_properties['bank']
statement_type = statement_properties['statement_type']
new_format = statement_properties['new_format']

print(bank, ' - ', statement_type, ' - ', new_format)

BankType.NU  -  StatementType.DEBIT  -  None


In [3]:
#extracted_words = doc_processor.get_extracted_words()
#
#if statement_type == 'debit':
#    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_extracted_words.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_extracted_words.csv')
#    
#extracted_words.to_csv(extracted_words_path, index=False)

In [4]:
corrected_extracted_words = doc_processor.get_corrected_extracted_words()
corrected_extracted_words

,page,text,x0,top,x1,bottom
0,1,Luis,424.700,44.910,443.340,54.910
1,1,Fernando,445.990,44.910,490.560,54.910
2,1,Laris,493.210,44.910,515.070,54.910
3,1,Pardo,517.720,44.910,545.000,54.910
4,1,Cuenta,424.010,59.910,457.860,69.910
...,...,...,...,...,...,...
1186,7,099,342.736,803.528,358.104,811.528
1187,7,1133,360.224,803.528,376.272,811.528
1188,7,7,522.800,815.528,527.000,823.528
1189,7,de,529.120,815.528,538.680,823.528


In [5]:
#if statement_type == 'debit':
#    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_corrected_words.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_corrected_words.csv')
#    
#corrected_extracted_words.to_csv(corrected_words_path, index=False)

In [6]:
from models import TableProcessingFacade

table_processor = TableProcessingFacade(corrected_extracted_words, statement_properties)
reconstructor = table_processor.get_reconstructor()
structured_table = reconstructor.get_structured_table()
structured_table

,date,description,MONTO EN PESOS MEXICANOS
0,31 AGO 2024,FECHA DEL 01 AL (31 DÍAS) MONTO EN PESOS MEXIC...,None
1,23 AGO 2024,Depósito en Cajita: Mi Primera Cajita,"-$32,500.00"
2,23 AGO 2024,LUIS FERNANDO LARIS PARDO Inversion,"+$32,000.00"
3,None,"Depósito SPEI, Hora: 06:13:03, Recibido de BBV...",None
4,None,cliente LUIS FERNANDO LARIS PARDO (Dato no ver...,None
5,None,"por esta institución), por concepto Inversion....",None
6,None,"012910015285049182 clabe, Clave de rastreo",None
7,None,"MBAN01002408230078163477, Clave de referencia ...",None
8,23 AGO 2024,LUIS FERNANDO LARIS PARDO Inversion,+$500.00
9,None,"Depósito SPEI, Hora: 06:11:51, Recibido de BBV...",None


In [7]:
reconstructed_table = table_processor.reconstruct_table()
reconstructed_table

,date,description,MONTO EN PESOS MEXICANOS
0,23 AGO 2024,Depósito en Cajita: Mi Primera Cajita,"-$32,500.00"
1,23 AGO 2024,LUIS FERNANDO LARIS PARDO Inversion Depósito ...,"+$32,000.00"
2,23 AGO 2024,LUIS FERNANDO LARIS PARDO Inversion Depósito ...,+$500.00
3,20 AGO 2024,Paola Arias Transferencia 2024 Transferenc...,"-$1,100.00"
4,20 AGO 2024,Retiro de Cajita: Mi Primera Cajita 2024,"+$1,100.00"
5,13 AGO 2024,Depósito en Cajita: Mi Primera Cajita,"-$19,000.00"
6,13 AGO 2024,LUIS FERNANDO LARIS PARDO Inversion Depósito ...,"+$19,000.00"
7,12 AGO 2024,Luis Laris bbva Transferencia Transferencia S...,"-$5,000.00"
8,12 AGO 2024,Retiro de Cajita: Mi Primera Cajita,"+$5,000.00"
9,11 AGO 2024,Depósito en Cajita: Mi Primera Cajita Nu Méxi...,"-$4,124.00"


In [8]:
#if statement_type == 'debit':
#    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_reconstructed_table.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_reconstructed_table.csv')
#    
#reconstructed_table.to_csv(reconstructed_table_path, index=False)

In [9]:
from models import DataProcessingFacade

data_processor = DataProcessingFacade(corrected_extracted_words, reconstructed_table, statement_properties)
period = data_processor.get_period()
print(period)

(datetime.date(2024, 8, 1), datetime.date(2024, 8, 31))


In [10]:
df_transactions = data_processor.get_transactions()
df_transactions

,date,description,amount,type
10,2024-08-11,MANUEL ANTONIO CASTILLO POLANCO Transferencia ...,3834.00,Abono
11,2024-08-11,Saldo inicial,519415.93,Saldo inicial
7,2024-08-12,Luis Laris bbva Transferencia Transferencia S...,-5000.00,Cargo
6,2024-08-13,LUIS FERNANDO LARIS PARDO Inversion Depósito ...,19000.00,Abono
3,2024-08-20,Paola Arias Transferencia 2024 Transferenc...,-1100.00,Cargo
1,2024-08-23,LUIS FERNANDO LARIS PARDO Inversion Depósito ...,32000.00,Abono
2,2024-08-23,LUIS FERNANDO LARIS PARDO Inversion Depósito ...,500.00,Abono


In [11]:
#from models import CsvExporter
#
#exporter = CsvExporter(df_transactions)
#if statement_type == 'debit':
#    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_normalized_table.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_normalized_table.csv')
#
#exporter.export_data(normalized_table_path)